# Data Scraping

This section documents how the raw data is collected. The goal is to make the data scraping step reproducible and to show which files are created before the data is processed further.

We collect three types of data: electricity prices, electricity consumption, and weather. The data is fetched automatically from public APIs and saved as raw files in the `data/` folder. These files are kept separate from the later processing step, so the collection process remains easy to inspect and rerun.

---
## Data sources

The scraping pipeline collects four raw datasets. The table below gives a short overview of what is collected and where it comes from.

| Source | API | What we collect |
|--------|-----|-----------------|
| Day-ahead electricity prices | Energi Data Service | Hourly DK1 prices |
| Electricity consumption | Energi Data Service | Hourly electricity consumption by grid area |
| Historical weather observations | Open-Meteo Archive API | Hourly observed weather variables |
| Previous-run weather forecasts | Open-Meteo Previous Runs API | Weather forecasts as they were issued 1-5 days before each target hour |

### 1. Day-ahead electricity prices

The price data is fetched from **Energi Data Service**. Two overlapping datasets are used: `Elspotprices` for older history and `DayAheadPrices` for more recent observations. The scraper standardises the column names, keeps the DK1 price area, combines the two sources, removes duplicate hours, and saves one hourly price file.

The output contains timestamps, the DK price area, and prices in both EUR/MWh and DKK/MWh.

In [ ]:
from src.data.data_collection import fetch_day_ahead_prices

prices = fetch_day_ahead_prices(
    start="2021-01-01",
    end="2026-04-28",
    price_area="DK1",
)

### 2. Hourly electricity consumption

Consumption data is also collected from **Energi Data Service**. The raw records are reported at grid-area level. During collection, the relevant grid areas are kept and the data is saved as an hourly consumption file for DK1.

In [ ]:
from src.data.data_collection import fetch_consumption

consumption = fetch_consumption(
    start="2021-01-01",
    end="2026-04-28",
    price_area="DK1",
)

### 3. Weather actuals

Historical weather observations are fetched from the **Open-Meteo Archive API** for a representative DK1 location, using latitude 56.15 and longitude 8.45. The scraper requests hourly weather data for the full project period and saves the returned variables in one CSV file.

| Variable | Why it matters |
|----------|----------------|
| Wind speed and direction at 10 m and 100 m | Wind turbine production proxy; 100 m is closer to actual hub height |
| Shortwave radiation | Solar PV production proxy |
| Cloud cover | Affects how much solar radiation reaches panels |
| Temperature at 2 m | Drives heating and cooling demand |
| Mean sea level pressure | Captures large-scale weather regime shifts |

The file contains one row per hour and region, with the variables shown above.

In [ ]:
from src.data.data_collection import fetch_weather_actuals

weather_actuals = fetch_weather_actuals(
    start="2021-01-01",
    end="2026-04-28",
)

### 4. NWP weather forecasts

Numerical weather prediction (NWP) forecasts are fetched from the **Open-Meteo Previous Runs API**. These are *as-issued* forecasts: they show what the ECMWF forecast looked like before the target hour occurred. For each target hour, the API provides the forecast as it looked 1, 2, 3, 4, and 5 days ahead, stored as `_previous_day1` through `_previous_day5` columns.

The previous-run API only provides forecast archives from 2025 onwards, so this part of the scraper uses a shorter date range than the other sources.

In [ ]:
from src.data.data_collection import fetch_weather_forecasts

weather_forecasts = fetch_weather_forecasts(
    start="2025-01-01",
    end="2026-04-28",
)

---
## Running the full data collection

The four functions above can be run separately when only one raw file needs to be refreshed. To rebuild all raw inputs in one run, the same functions are wrapped in `fetch_all()`.

In [ ]:
from src.data.data_collection import fetch_all

results = fetch_all(start="2021-01-01", end="2026-04-28", price_area="DK1")

After the calls complete, the `data/` folder contains the following raw input files. These files are intentionally kept close to the API outputs; the heavier transformations are handled in the next processing step.

| File | Created by | Content | Coverage |
|------|------------|---------|----------|
| `day_ahead_prices_dk1_raw.csv` | `fetch_day_ahead_prices()` | Hourly DK1 day-ahead electricity prices | 2021-present |
| `consumption_dk1_raw.csv` | `fetch_consumption()` | Hourly DK1 electricity consumption | 2021-present |
| `weather_actuals_raw.csv` | `fetch_weather_actuals()` | Hourly weather observations for DK1 West | 2021-present |
| `weather_forecasts_raw.csv` | `fetch_weather_forecasts()` | NWP previous-run forecasts for DK1 West | 2025-present |

---
## Next step: Data processing

The scraping step only downloads and stores raw data. The next step is handled by `src/data/data_processing.py`, where the raw files are cleaned, combined, and transformed into the files used later in the project.

For the weather data, the processing step creates two additional files:

| Processing output | How it is created |
|-------------------|-------------------|
| `weather_error_distributions.csv` | Previous-run forecasts are compared with weather actuals at 24, 48, 72, 96, and 120 hour horizons |
| `forecast_dataset.parquet` | A 120-hour forecast dataset is generated for each 12-hour issue time from 2021 onwards |

This keeps the workflow separated into two clear stages: first collect the raw API data, then process it into analysis-ready files.